# Quest Part 2 Step 2

Before we can do SLOOH, we will need to install some packages. Unfortunately, it isn't as simple as just some import statements; the package versions need to be compatible with each other. 

Follow the steps below carefully.

1. **Run this cell below.**

In [ ]:
!jupyter lab clean
!pip uninstall -y jupyterlab_widgets ipympl
!pip install jupyterlab_widgets ipympl astropy

2. **Then, shut down the kernel (in the upper toolbar, Kernel -> Shut Down Kernel)**

3. **Reload your Jupyter tab (Ctrl-R) and come back to here without running any cells**

4. **Run the two cells below**

In [ ]:
%matplotlib widget

In [ ]:
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
import astropy.units as u
import re

5. **If you haven't already, upload your FITS file to the same folder as this page.**

6. **Copy the file name and place it here (don't include any slashes (`/`) or anything before that)**

In [ ]:
# Change the file path for the variable below.

v_band = ...

7. **If you've done it correctly, this following cell should print out an interactive graph of your image.**

In [ ]:
# Opening the V Band

# Each section (HDU) contains data (like an image) and a header with metadata.
v_list = fits.open(v_band)

# Print a summary of the file’s contents: how many HDUs it has, their types, and data shapes.
v_list.info()

# Read the data and header from the first HDU.
# The data usually contains the image, and the header holds metadata about the observation.
v_data = v_list[0].data
v_header = v_list[0].header

# Display the header, which contains metadata like the telescope used, exposure time, etc.
v_header

# Apply BSCALE and BZERO if they exist in the header.
# These are used to scale the raw pixel values into meaningful physical units.
v_data = v_data * v_header.get('BSCALE', 1.0) + v_header.get('BZERO', 0.0)

# Display the image data as a 2D NumPy array of pixel values.
# This array represents the brightness of each pixel in the image.
v_data

# Print the Python type of the image data to confirm it's a NumPy array.
print(type(v_data))

# Print the dimensions of the image (rows, columns),
# which tells you the resolution of the image in pixels.
print(v_data.shape)

# The x and y axes represent pixel coordinates (not sky position).
# The color scale shows the brightness at each pixel, higher values = brighter regions.
plt.imshow(v_data, cmap='gray', origin='lower')
plt.colorbar()  # Adds a color scale bar to interpret pixel intensity values

Now that you've opened your FITS file, return to the Quest, Part 2 Step 2 and you'll learn which star is which.
# Return to Part 2 Step 2 of the Quest

# Quest Part 2 Step 3

In the next section, you'll begin by running the first cell. This will bring up a plot where you'll be asked to select two stars. Before making your selections, feel free to zoom in on the plot using the square zoom tool for better precision. Once you're ready, click on the stars starting with Star A (the brighter or larger star), followed by Star B.

In [ ]:
# Set up WCS from header
wcs = WCS(v_header)

fig, ax = plt.subplots()
ax.imshow(v_data, cmap='gray', origin='lower')
ax.set_title("Zoom/pan first. Then press 's' to select stars.")
ax.set_xlabel("Pixel X")
ax.set_ylabel("Pixel Y")

clicked_points = []
click_mode = {'enabled': False}  # Mutable so handler can update it

def on_key(event):
    if event.key == 's':
        click_mode['enabled'] = True
        ax.set_title("Selection mode active — Click on Star A, then Star B")
        fig.canvas.draw()
        print(" Now click on two stars to select them.")

def on_click(event):
    if not click_mode['enabled']:
        return  # Ignore clicks until 's' key is pressed
    if event.inaxes:
        x, y = event.xdata, event.ydata
        clicked_points.append((x, y))
        ax.plot(x, y, 'ro' if len(clicked_points) == 1 else 'bo')
        ax.text(x + 5, y + 5, f'({x:.1f}, {y:.1f})',
                color='red' if len(clicked_points) == 1 else 'blue')
        fig.canvas.draw()

        if len(clicked_points) == 2:
            fig.canvas.mpl_disconnect(cid_click)
            fig.canvas.mpl_disconnect(cid_key)
            ax.set_title("Stars Selected")
            ax.legend(['Star A', 'Star B'])
            fig.canvas.draw()

cid_key = fig.canvas.mpl_connect('key_press_event', on_key)
cid_click = fig.canvas.mpl_connect('button_press_event', on_click)
plt.show()

In [ ]:
x1, y1 = clicked_points[0]
x2, y2 = clicked_points[1]
print(f"Star A coordinates: {x1:.1f},{y1:.1f}")
print(f"Star B coordinates: {x2:.1f},{y2:.1f}")

Now that you've selected both stars and see their coordinates, copy them or write them down on a piece of paper, and then return to the Quest, Part 2 Step 3.
# Return to Part 2 Step 3 of the Quest


# Quest Part 2 Step 4

After clicking your two stars, extract them here by running the cell below. The code will calculate the distance between the two stars and you will need to input the angular separation in the Quest.

In [ ]:
if len(clicked_points) < 2:
    raise RuntimeError("Please click two points on the plot before running the next cell.")

x1, y1 = clicked_points[0]
x2, y2 = clicked_points[1]

# Then do your calculations below:

# Calculate the distance between the two stars in pixel units.
pixel_sep = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
print(f"Pixel separation: {pixel_sep:.2f} pixels")

# Get the instrument information from the FITS header.
instrume = v_header.get('INSTRUME', '')
match = re.search(r'@ ([0-9.]+)\s*arcsec/pix', instrume)
if match:
    unbinned_pixel_scale = float(match.group(1))
    print(f"Unbinned pixel scale from header: {unbinned_pixel_scale:.3f} arcsec/pixel")
else:
    raise ValueError("Pixel scale not found in INSTRUME header.")

# Get the binning factors used during image capture.
x_bin = int(v_header.get('XBINNING', 1))  # 'XBINNING'
y_bin = int(v_header.get('YBINNING', 1))  # 'YBINNING'

if x_bin != y_bin:
    print(f"Warning: Unequal binning detected (X={x_bin}, Y={y_bin}). Averaging.")
binning_factor = (x_bin + y_bin) / 2.0

# Adjust pixel scale by binning factor.
pixel_scale = unbinned_pixel_scale * binning_factor
print(f"Adjusted pixel scale (after binning): {pixel_scale:.3f} arcsec/pixel")

# Convert pixel separation to angular separation.
angular_sep = pixel_sep * pixel_scale
print(f"Approximate angular separation: {angular_sep:.2f} arcsec")


# Return to Quest Part 2 Step 4

# Quest Part 2 Step 5
In the next section, you'll begin by putting the angular separation listed in the Quest from the SIMBAD database into the cell and run it. Then, running the cell after that will calculate the percent difference between your measurement and the one from the SIMBAD database.

# Edit this cell by putting the angular separation listed in the Quest from SIMBAD for your double star system. Enter the number only.

In [ ]:
#Get the angular separation listed in the Quest from the SIMBAD database
SIMBAD_angular_sep = 

In [ ]:
#Calculate the percent difference
percent_difference = (np.abs(angular_sep-SIMBAD_angular_sep)/SIMBAD_angular_sep) * 100. 

print(f"The percent difference between your measurement and \
the one from the SIMBAD database is {percent_difference:.1f}%")

# Return to Quest Part 2 Step 5

# Quest Part 2 Step 6

In the next section, you'll find the actual distance between the two stars in your double star system. First, you'll need to enter the distances to each star in the next cell. Then, running the cell after that will calculate the actual distance to each star and provide the result in light-years and AU.

[This part of the code is where they'll calculate the distance between both stars. Have them enter the angular separation, but have them type in the variable name instead of a number. Display the physical distance results in light-years and AU] We'll include a table that gives students the distance to each star in the Quest so that they can enter the values in here. The code should display the true seperation in light-years and AU.

# Edit this cell by putting the distances to each star in your double star system. Enter the numbers only.

In [ ]:
#Get the distances to each star in light-years listed in the Quest.
distance_star1 = 
distance_star2 = 

In [ ]:
#Calculate the actual distance between both stars.
actual_distance_ly =np.sqrt((distance_star1)**2+(distance_star2)**2- \
                            2.*distance_star2*distance_star1* \
                            np.cos(angular_sep*(1./60.)*(1./60.)*(np.pi/180.)))

print(f"The actual distance between each star is {actual_distance_ly:.2f} light-years")

actual_distance_AU = actual_distance_ly * (63241.1/1.)
print(f"The actual distance between each star is {actual_distance_AU:.0f} AU")

# Return to Quest Part 2 Step 6